# F2 — Preprocesamiento, limpieza, transformación y validación
**Proyecto:** Análisis del desempeño SIMCE 2025 — 4° Básico, Matemática

## Objetivo
Construir un pipeline **reproducible, modular, auditable y trazable** que:

- obtenga la base SIMCE 2025 desde `data/raw/`;
- explore el conjunto **antes de modificarlo**;
- limpie y estandarice tipos sin imputar información institucional o resultados;
- identifique y documente registros **Efectivos** y **No Efectivos**;
- incorpore `DimGeografia` **dentro del código**, sin depender de un CSV externo;
- valide casos normales, límites y excepciones;
- genere el dataset procesado para F3;
- genere evidencia del filtro (`auditoria_filtro.csv`) y de cobertura territorial (`cobertura_regional.csv`).

### Alcance de F2
Esta fase prepara y valida datos. **No realiza visualizaciones, pruebas de significancia, modelos predictivos ni priorización territorial**; esas tareas corresponden a fases posteriores.

### Fuente
Base pública SIMCE 2025 de la Agencia de Calidad de la Educación. El archivo bruto debe mantenerse sin modificaciones en `data/raw/`.

## 1. Configuración y reproducibilidad

Se utilizan rutas relativas al repositorio. El archivo procesado principal incorpora una marca de ejecución `AAAAMMDDHHMM`.  
La celda también muestra las versiones del entorno para dejar evidencia reproducible.

> Antes de entregar: ejecutar **Restart Kernel → Run All Cells** y conservar las salidas visibles.

In [1]:
from pathlib import Path
from io import StringIO
from datetime import datetime
import base64
import platform
import sys
import warnings
import zlib

import pandas as pd
from IPython.display import display

try:
    from zoneinfo import ZoneInfo
except ImportError:
    ZoneInfo = None


def encontrar_raw_dir() -> Path:
    """Localiza data/raw usando rutas relativas compatibles con raíz o carpeta F2."""
    candidatos = [
        Path.cwd() / "data" / "raw",
        Path.cwd().parent / "data" / "raw",
        Path.cwd() / "F2" / "data" / "raw",
        Path.cwd().parent / "F2" / "data" / "raw",
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError(
        "No se encontró data/raw. "
        f"Directorio de ejecución: {Path.cwd()}"
    )


def buscar_simce(raw_dir: Path) -> Path:
    """Busca el CSV SIMCE 4° básico 2025 dentro de data/raw."""
    patrones = [
        "simce4b2025_rbd_final.csv",
        "*simce4b2025*.csv",
        "*simce*2025*.csv",
        "*simce*.csv",
    ]
    for patron in patrones:
        archivos = sorted(raw_dir.glob(patron))
        if archivos:
            return archivos[0]
    raise FileNotFoundError(f"No se encontró un CSV SIMCE en {raw_dir}")


def obtener_fecha_ejecucion() -> datetime:
    """Obtiene fecha/hora de ejecución intentando usar America/Santiago."""
    if ZoneInfo is not None:
        try:
            return datetime.now(ZoneInfo("America/Santiago"))
        except Exception:
            pass
    return datetime.now().astimezone()


RAW_DIR = encontrar_raw_dir()
PROCESSED_DIR = RAW_DIR.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SIMCE_FILE = buscar_simce(RAW_DIR)
FECHA_EJECUCION = obtener_fecha_ejecucion()
MARCA_EJECUCION = FECHA_EJECUCION.strftime("%Y%m%d%H%M")

OUTPUT_FILE = PROCESSED_DIR / f"simce4b2025_matematica_efectiva_{MARCA_EJECUCION}.csv"
AUDITORIA_FILE = PROCESSED_DIR / "auditoria_filtro.csv"
COBERTURA_FILE = PROCESSED_DIR / "cobertura_regional.csv"

print("=" * 72)
print("CONFIGURACIÓN")
print("=" * 72)
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("Sistema:", platform.platform())
print("Entrada:", SIMCE_FILE)
print("Salida principal:", OUTPUT_FILE.name)
print("Auditoría:", AUDITORIA_FILE.name)
print("Cobertura:", COBERTURA_FILE.name)

CONFIGURACIÓN
Python : 3.14.7
Pandas : 3.0.5
Sistema: Windows-11-10.0.26200-SP0
Entrada: C:\Users\emper\f1_s01_evaluacion_entregable_grupo7\data\raw\simce4b2025_rbd_final.csv
Salida principal: simce4b2025_matematica_efectiva_202609142338.csv
Auditoría: auditoria_filtro.csv
Cobertura: cobertura_regional.csv


## 2. Catálogos y regla de efectividad

Los códigos administrativos se traducen a etiquetas comprensibles sin borrar las claves originales.

### Observaciones al puntaje
El proyecto utiliza el catálogo `OBS_PUNTAJE` para interpretar `marca_mate4b_rbd`. La variable representa una observación asociada a la posibilidad o representatividad de reportar el resultado.

**Punto de control importante sobre la marca 2:** existen casos en los que `marca_mate4b_rbd = 2` puede coexistir con un puntaje numérico. Por ello, F2 **no oculta esos casos**: los contabiliza, los deja explícitos en `auditoria_filtro.csv` y muestra una advertencia.

La glosa utilizada por el proyecto para el código 2 es *“Por causas ajenas a la Agencia, los resultados no son representativos del desempeño de los estudiantes”*. Antes de la entrega definitiva, el equipo debe conservar como respaldo el diccionario/glosa oficial de la **base 2025 específica**. El notebook no presenta como “confirmada 2025” una glosa que no esté respaldada por ese documento.

La regla operacional heredada del flujo del proyecto es:

- alumnos evaluados `> 0` **y**
- ausencia de observación al puntaje

→ **Efectiva**.

Todo lo demás queda como **No Efectiva** y se audita antes de excluirse.

In [2]:
DEPENDENCIA_1 = {
    1: "Municipal Corporación",
    2: "Municipal DAEM",
    3: "Particular subvencionado",
    4: "Particular pagado",
    5: "Corporación de administración delegada",
    6: "Servicio Local de Educación",
}

DEPENDENCIA_2 = {
    1: "Municipal",
    2: "Particular subvencionado",
    3: "Particular pagado",
    4: "Servicio Local de Educación",
}

GSE = {
    1: "Bajo",
    2: "Medio bajo",
    3: "Medio",
    4: "Medio alto",
    5: "Alto",
}

RURALIDAD = {
    1: "Urbano",
    2: "Rural",
}

OBS_PUNTAJE = {
    1: "No es posible reportar resultados, porque la cantidad de estudiantes evaluados es insuficiente. (6 o menos)",
    2: "Por causas ajenas a la Agencia, los resultados no son representativos del desempeño de los estudiantes",
    3: "Por causas ajenas al establecimiento, los resultados no son representativos del desempeño de los estudiantes",
    4: "La aplicación de la prueba extendida no permite evaluar esta asignatura.",
}

# DimGeografia: 346 comunas embebidas dentro del código.
DIM_GEOGRAFIA_B64 = """eNq1XDtz2zoW7vdXsNvWIqmHB5XtxEnuOI7X8Xrm3mYHohAJWZBQQNIzdnd/wrbbuUyRIpPuznb8Y/sdQPJTLwJUYxMUz3dA8ODgvIBMT+RU/8uIqdQFW/zL3M250TeyyCRnj1eLnzKd1wW17L8/NP585JnRd3T1yUxE8bceu+KGz3nW3LNej334VstvtcDlwWPjXJtKPFJGrh1voB2yI1Xp6L0u5zKTejeEFI2cm3rKFRrgf4GHoyOV66ISXhAxO+G5bH5yL+qEnWiVzXjhxzxl72v86EXbZxcyW08as6Oi0l/4lJcVZ3HvRfPg+Y3VKMlWlJh9FF+lUroQpTdIwj5LYQyP3mkz8e8LjY+qMDg7AsTsrYrONF1hNE64wuj60Mbsk1J82vxP+FDj5XkRXYiJ0dFEREcVz1r0I2FXOtNzfAFq4DUem34I+KDcND949FaJYj1IwpYdTTB0IOfz5heuD560VtOm62lpKqqJWDsfNpDiLZwIHWGGbHj9ZxDgN+OY+QYyk1gZeGi2p47ZGymm7huqnE+N3gUkIQVQZhpX4H/NFY26aUsZO1U6ESo64SYXRVuAhJ0aIY0seFvKdHm9ji7Fd4LSz8eapT3IPa5xgZc949FnYVYJ2YkoKswGd7u/DiF+vO0HgDcqJjzT0F6eCCm9xHs5rVfJ7G4Q0OJczmpe+PZhwK5lVq9avzbQk+xqrCu4ojUc82UulC8ATdtCKO8OJOxMl9G1VLr0hUihREl9F1mrXoCzzEnd4RLj8OmG5p83AAlkPuZmzBWWa1+UhH0kWya64JWR/m+Tsou6kBUnKfHF6LPL5gcstNpUfLJVPvukvuZQh82PUrN+70Xz4PkNR7wObbAVjYSu5GNlP3koGFlxRdb8KoKRUvYb5nJ0KkzR3BcTcReMCAVRZ7Mar3mDrxKKNmT/qCXEy4R/gUMoHqgdu+rAZmiLF7MPpSJqSHqZ1Rx3SBU9v9cS02kSaHUYo32a0Y/NMCRrmSgRnXEzDe1Vwi7hgIFsEoqUWtPxbVmJMW8tulAQotIGs6efutX4DAsZD4KhYRpLNdFBKAmWxHkdCPLYCEHpsz+wwmB5bC3efZposCxgYvf7GN+HVhBOTN8Jokj2RhAQTDj5tcaiXYbhWAvoxNR3YTADLC5wa9vP04GdAuTQFJKaGOmnN0LRyLCecmP0OBwLiwzHQjolizcUKyUHEgNIhncoVp+wrngHb2hvwA95o3NZTFvjDS3eqVByLqwbBSWZcdJJ/eHiu7ofOwUmnVWJvO4UFIpacXJDbzuFhW7ixbdazOpux4BsjIqLYtLtN3PywCMXUWgLPaJIxNRSY83tjxZKdF4334OhYmvmZjMRjJSwTyrvokspzCkITHSkBPyY7aM1wCuMhbXMTfROFFgSVHQMu5ObiY4+/R1u6VQWJRtQaCub8bnmCg2M4iVffKLNLEbeLMgRmoh9ciCDnYyovTEgn0hVvN4jiz57o5ufcvtc9mcxYO8MXHOjy72xGMIAKGGSjA3fHxOaLGiqrQ6QP4tDQMIc2ZvI9g7g15cVv0NrXyx60EXyZqut6s8ghm1c53ubEr2EUhozuVfd0UudC2zdzCupMr2/Od7rs0ux3RLyxx8A/1vd/Cj2KLhuib+WmaAUW2dcyO6i7CK3MWuNG5TNo8+vtltjYXysFwW32QhKFOyTE4XTqlpsNzTC2KRkTGDkOl1LVvDps3N+Iyd8slcuA1i5RkxsSrEzPstELWmWQfLgTuDB7VavPxuKUTff5zLje2OB5kxS2Fl3qmhecIGfr5VWe8MnyeraKn3BgsRK5bLTKf+CxRDrIx6luMbeeIzYheLZXl/jEO5gbh3NfbGANeRcwx1CR6O/DfFTrQQbUjmEwlwakhvjLtfRfq4NO1xJSd5JUVbQy5ncFPJfj4B3qY2YV9qDNmVv87mYGD7xoe4v7rWnhPQLhUXChyuJNSbnptVlPfHIJZFOFDc+vA+toj6BTbDR/FgL4CQtuuRfuFC7ktOiVEP8aQUauuKQZdMPwdYKFLsP/gvyhD6eqlt8ACuiMmt+4ZL6v2j4UMeU4Ve8+bkhyLGJnmygDJO9+V540cO40UoW3IsYBvhGJ34TLYxrnZNC86Ims3mKac6jU45VR/r1f8SuROHXfcrPZTNIUYuBxzKPkTYkdTYptGj4UJOaVePmLz/elOkqphtTnpvIKVBqWny5Z8TktFVyd231jNgFxX+DmSyMF8DwMQ44NcILYsR+F2bMy+gYGoRv+3wjdiz1GDqajXo2Dy7mbmEcuWquxxv0/EqQXm8jCokCzFPoX18Ea+TCdLjltAj4oqTsVGkD/8UboW91IdVU+CLAYKKcly+5W4q1N/3oebUh5OyC33mjHS6sOKp58H4nrNB2sXZFUL4gVPmYY4XypXer3Jx05Y4QMTsydm0ZkYVwJsa1D+XDtQctZRSbn2L3GfGUmCLcRVWr3It1n1YaXkhlrUIPgIEtZWj+VDzXpQ/AkF1J0/y1s9glTy4XlSHNn8VUUNLZB8KW0bb4cM+I6cuNDVXh+FBT0v2r34vDgajhhrUQ82fkA7jpmcwlHvbr+pCd07K2u8g+ox7ZDCD3HPRDV3Ig/MiXjoQubZ7UB6G3UJfHzT2WZ+P1BaGorurMlmv6UCfs9zofexKnrsx4eWcTxCGFWO1k5QXlfg+tD1U1PwpckvMu8s1zJ94CYeOGW5TPNgwyl4ugXqRWDRojwjrSd+XXIRAD9o6rG25kEfI+Q/ZOm7HgARAj/FpX3IR045Cs/0xT2NwbhBJutiwhSM4wZ89rccOjD/lcGMlVAFQMp2RiyOiihGnJywAsiggYChAEvRzcJFmZ+ovzEf1xbI3qZg9hGwT5TfyGAmMBILAJhJ5oejyb5XJSBWDB89aqChsXcsFVRj6wN0Z84PxBYyiJ4A9Dm1soAKtbSnBs8+K0Zh7Gds/YNAjBRgUwK2v8CYCxgQ6eLV7PHydlb42Lpftj9ElZUf08vw1AcfbwZ/wqygCYITurc95WKTyDGGEym9Zy/wzikNLeXIZ0g7xCw+HzB/UE6vtaZpWmnRSbQMCNhv+MT+GM2M1JivIfNqGKJmWma2Fg+9DejJdaxWKkWzFsDfn49Wq0IzklULIZz5tffvQp2Rjlq3HYkZoslLqyisgPwMn3R4zia39vR4jh0xteCCMbLlOvdcaO9IdLMbjmhu/2FhT5l0o33+naZhLKymgPUvI3s3riQUnBM5hTM+lB6wzczAaLrpvvStx5gPTZGxfgER7EA4w5DB01eW1f7EJOG16EVK+1yC7E5G8KyIsXZ3I3Re417PiNapQw3XeTlIR9KrWB5Y9L2sfmGq0p46V4f8qqV4vZDvQJrRzGJY5bE9OGtVsx8yFd7FOjyILHWy8i9rRty0VET6hOrz2Oq5O64GO1WycoR2H3WOPS7T2W1a5i+ow2ZqdwuJSovzR/tadOXHoPOu5V9HQH6ofGakoYjLclXspuEluWB8BxV8D7MG7um5/ibrFtOZ9jGHsUl73FSLgzIlwCYtl+zuKohiKlkxD6HbChirAp1Lowk075xAsCujxYUu+NAXSILItX9mN3DBL2roaUiox3yoOioXMI/z2mj+EV3bEfPptR9fK+OcXssUxlz6zorAVTvYq+BfFJ2ZOnDG0M6/WcPpFKRPj7yuDcA7vYaeAlxQaGZJ1PaUcd5DS6XWjcowLjZCp4uJHtNikXeKyPT1LTGuEUPj0yNqu5hscgiIdVBDXwjt3O4r0wSdxobdQ2YRzcztB3Rkzh+uhOeNh8x4rb1qYdWwPxPa2F5X65Pf2hE04PJ4eQlJ/WGDG6aU8WMjeikGafXGBxGZnzG2H2+jJ0O+f1v9drnlZcUtb8V1VAjN6Wc3S9uLN3MWbnvOJKlHvlEpMShStr3/GCy2LTHErYRwF/a66VrHhha/4pByM5jUziEjLLxsGT5quSiWcwbNgSGSszPgvVhZZdQycWGtYvv1lVYxQGbpPE63behEHbbafHulxZZRoGPWBvYcHb1L6L86wqBApjMYTVLLIZfONx54M+Yh+KiZgL/LEHtXUMbwNmMBQrqtLtGJxcBR4tq2w6xraHFNCGtq+dQ9sdIRe0K6j7IUkI+1LI7pFTt/dOrz7gIgybotrRMTcFbV7pvOcUDbRq/WvXqrY3JOiL1fXVYch2F2LdtSrp0c5DOYfr3i1ufMCa/9TNT931x4O16erHjqa1xEIMFWs6n+wxbTVsfnKl1cry1TDwBGaXvpF70a8xBbImFOzsWlJid2aGzGrTeacHy02R59rknS+X8ZBdikwrUXXe8RGlmrLOYd22g980t7srO0ZPXCnRRzo4ruuhds3okhJFnfc7Ztd4qhP5szXBEyrHohCCPX3xoqYtHhHVGHUPD30iTRe25ivkZCEqZfOdHiVlHt5/ytnwjNt0YS9x4TDVhQnxEpgsn3zePS65nPDewsU7ZW5EcbXYtbncXtYpdMyOoQA7haQIIEmJ3WjOi2m3HabIO/m+oZh9VyZliyDQwhg/trvGpuOUZisPSwkEXtaD3HTgp77ETpcn4pJ/oDuH7z9W5AdjD6iSnk/tJgm03CbJRbtrbHuGsT2qsnPo5OEYwG6U6Uv8dFGItzzYsmv8vrNdv8AN3gE7tdk1igiXrJfaYxYnkmI5vXRxaKZrrsiwJVuo7VplXhUt7kRLbmuxulhlG2n6JF/oQY4Z0tx/kV69HtBkld9qWXgN2JCUqlpRYb4T9YiO54JTtKq6bTVAbA9j0nTlzmD8ZyHXVBtspLUpYKML3ZoycZnPy52/9VPi1CUyaJvZOuI+O6LKxeg2uqDiZIwsfIBev+du09XB8nr1CdK9nTDsKf50BsLaY+jX41Bd7tOmNUUrI8KB4odkmZsMawEH5KuPFa4HPfYGfoF0VUo9e2AXxcRVc795J2XvYDMMGThq2w7mbSDJQ1+ia7kyeNMCzB5iuDivPASH1G2+bZPvNpABW7O8t8AYug0m2w4V2Abj9uh9mEKQZFiHDtnvdTHdcAjhCpCYfag4ya89kAcvZOS2fecbEGgFGsPtWu0z7oZBlY9i46FAWwBSdi6LgJfoUy6wEnfw1rUvxgCq8p5qCZUvwpBdGToYq52gJ/YccGtsPhx74/L7/ij0UUkoQnqS4Ae5fW/6RgyX5T7lY9lSOT6HcRb4ucw0FNvWYfk/+Z8rsQ=="""


def construir_dim_geografia() -> pd.DataFrame:
    """Reconstruye y valida DimGeografia embebida sin archivos externos."""
    datos = zlib.decompress(
        base64.b64decode(DIM_GEOGRAFIA_B64.encode("ascii"))
    ).decode("utf-8")

    dim = pd.read_csv(StringIO(datos), sep=";")

    for col in ["region", "provincia", "comuna", "Zona", "Macrozona"]:
        dim[col] = (
            dim[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    for col in ["codigo_region", "codigo_provincia", "codigo_comuna", "Orden"]:
        dim[col] = pd.to_numeric(dim[col], errors="raise").astype("Int64")

    dim["pais"] = "Chile"
    dim["ubicacion_region"] = dim["region"] + ", Chile"
    dim["ubicacion_provincia"] = (
        dim["provincia"] + ", " + dim["region"] + ", Chile"
    )
    dim["ubicacion_comuna"] = (
        dim["comuna"] + ", " + dim["provincia"] + ", " + dim["region"] + ", Chile"
    )

    # Renombrar claves para unir directamente con SIMCE y conservar esos códigos en la salida.
    dim = dim.rename(columns={
        "codigo_region": "cod_reg_rbd",
        "codigo_provincia": "cod_pro_rbd",
        "codigo_comuna": "cod_com_rbd",
    })

    claves = ["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]

    assert len(dim) == 346, (
        f"DimGeografia debería contener 346 comunas y contiene {len(dim)}."
    )
    assert not dim.duplicated(claves).any(), (
        "DimGeografia contiene claves geográficas duplicadas."
    )
    assert dim[claves].notna().all().all(), (
        "DimGeografia contiene claves geográficas nulas."
    )

    return dim.copy()


dim_geo = construir_dim_geografia()
print(f"DimGeografia validada: {len(dim_geo):,} comunas")
display(dim_geo.head())

DimGeografia validada: 346 comunas


,cod_reg_rbd,region,cod_pro_rbd,provincia,cod_com_rbd,comuna,Zona,Macrozona,Orden,pais,ubicacion_region,ubicacion_provincia,ubicacion_comuna
0,1,Tarapacá,11,Iquique,1101,Iquique,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Iquique, Tarapacá, Chile","Iquique, Iquique, Tarapacá, Chile"
1,1,Tarapacá,11,Iquique,1107,Alto Hospicio,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Iquique, Tarapacá, Chile","Alto Hospicio, Iquique, Tarapacá, Chile"
2,1,Tarapacá,14,Tamarugal,1401,Pozo Almonte,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Pozo Almonte, Tamarugal, Tarapacá, Chile"
3,1,Tarapacá,14,Tamarugal,1402,Camiña,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Camiña, Tamarugal, Tarapacá, Chile"
4,1,Tarapacá,14,Tamarugal,1403,Colchane,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Colchane, Tamarugal, Tarapacá, Chile"


## 3. Obtención de datos

Para mantener el pipeline eficiente solo se cargan las variables requeridas por F2 y F3.  
Se eliminaron del flujo las variables de estándares de aprendizaje (`palu_eda_*`) porque se decidió que **no forman parte del producto analítico final**.

No se modifica el archivo original.

In [3]:
COLUMNAS_SIMCE = [
    # Identificación
    "rbd", "dvrbd", "nom_rbd",

    # Geografía: solo claves; los nombres finales vendrán de DimGeografia
    "cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd",

    # Departamento provincial
    "cod_deprov_rbd", "nom_deprov_rbd",

    # Clasificaciones
    "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",

    # Matemática
    "nalu_mate4b_rbd", "prom_mate4b_rbd",

    # Control y efectividad
    "marca_mate4b_rbd", "noaplica",
    "codigo_bbdd", "fecha_bbdd", "grado", "agno",
]


def cargar_simce(archivo: Path, columnas: list[str]) -> pd.DataFrame:
    """Carga de manera selectiva la base SIMCE sin alterar el archivo bruto."""
    cabecera = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        nrows=0,
    )
    faltantes = sorted(set(columnas) - set(cabecera.columns))
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en la fuente: {faltantes}")

    return pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        usecols=columnas,
        low_memory=False,
    ).copy()


df_raw = cargar_simce(SIMCE_FILE, COLUMNAS_SIMCE)

print(f"Filas cargadas   : {df_raw.shape[0]:,}")
print(f"Columnas usadas  : {df_raw.shape[1]}")
print("Archivo bruto preservado: sí")

Filas cargadas   : 7,143
Columnas usadas  : 20
Archivo bruto preservado: sí


## 4. Exploración inicial — medir antes de decidir

La rúbrica exige que las decisiones de limpieza estén precedidas por evidencia.  
Por eso esta etapa se ejecuta **antes** de modificar tipos o filtrar registros.

Se revisan:

- dimensiones;
- tipos de datos;
- valores nulos;
- duplicados;
- cardinalidad;
- distribución básica de alumnos y puntaje;
- valores potencialmente atípicos mediante IQR.

Los atípicos **no se eliminan automáticamente**: un puntaje alto/bajo o un establecimiento con muchos alumnos puede ser un dato válido. El diagnóstico solo deja evidencia para justificar decisiones posteriores.

In [4]:
def diagnosticar_inicial(df: pd.DataFrame) -> dict:
    """Genera evidencia descriptiva sin modificar el DataFrame recibido."""
    trabajo = df.copy()

    resumen = pd.DataFrame({
        "tipo_original": trabajo.dtypes.astype(str),
        "nulos": trabajo.isna().sum(),
        "unicos": trabajo.nunique(dropna=True),
    }).sort_index()

    duplicados_fila = int(trabajo.duplicated().sum())
    duplicados_rbd = int(trabajo["rbd"].duplicated().sum())

    numericas_interes = pd.DataFrame({
        "nalu_mate4b_rbd": pd.to_numeric(
            trabajo["nalu_mate4b_rbd"], errors="coerce"
        ),
        "prom_mate4b_rbd": pd.to_numeric(
            trabajo["prom_mate4b_rbd"], errors="coerce"
        ),
    })

    atipicos = {}
    for col in numericas_interes.columns:
        serie = numericas_interes[col].dropna()
        if serie.empty:
            atipicos[col] = 0
            continue
        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)
        iqr = q3 - q1
        limite_inf = q1 - 1.5 * iqr
        limite_sup = q3 + 1.5 * iqr
        atipicos[col] = int(((serie < limite_inf) | (serie > limite_sup)).sum())

    print("=" * 72)
    print("DIAGNÓSTICO INICIAL")
    print("=" * 72)
    print(f"Dimensiones: {trabajo.shape[0]:,} filas x {trabajo.shape[1]} columnas")
    print(f"Filas duplicadas completas: {duplicados_fila:,}")
    print(f"RBD repetidos: {duplicados_rbd:,}")
    print("Potenciales atípicos IQR (solo diagnóstico):", atipicos)

    display(resumen)
    display(numericas_interes.describe())

    return {
        "resumen_columnas": resumen,
        "duplicados_fila": duplicados_fila,
        "duplicados_rbd": duplicados_rbd,
        "atipicos": atipicos,
    }


diagnostico_inicial = diagnosticar_inicial(df_raw)

DIAGNÓSTICO INICIAL
Dimensiones: 7,143 filas x 20 columnas
Filas duplicadas completas: 0
RBD repetidos: 0
Potenciales atípicos IQR (solo diagnóstico): {'nalu_mate4b_rbd': 303, 'prom_mate4b_rbd': 56}


,tipo_original,nulos,unicos
agno,int64,0,1
cod_com_rbd,int64,0,344
cod_depe1,int64,0,6
cod_depe2,int64,0,4
cod_deprov_rbd,int64,0,44
cod_grupo,float64,114,5
cod_pro_rbd,int64,0,56
cod_reg_rbd,int64,0,16
cod_rural_rbd,int64,0,2
codigo_bbdd,str,0,1


,nalu_mate4b_rbd,prom_mate4b_rbd
count,7143.000000,6579.000000
mean,29.954781,254.734762
std,29.617364,25.615483
min,0.000000,152.000000
25%,8.000000,237.000000
50%,22.000000,254.000000
75%,41.000000,272.000000
max,273.000000,341.000000


## 5. Estrategia de limpieza y tratamiento de nulos

### Decisiones
1. **No se imputan RBD, claves geográficas ni categorías institucionales.** Imputarlas inventaría identidad o pertenencia territorial.
2. **No se imputa `prom_mate4b_rbd`.** Un puntaje faltante no se reemplaza por media/mediana, porque alteraría el resultado educativo.
3. **No se imputa `nalu_mate4b_rbd`.** Los nulos se interpretan de forma conservadora como ausencia de alumnos que rinden la evaluación dentro del establecimiento, para la regla de efectividad.
4. Los textos se limpian eliminando caracteres de control, espacios repetidos y cadenas vacías.
5. Las variables numéricas se convierten de forma explícita. Si aparece texto inesperado en una columna numérica, el pipeline se detiene.
6. `fecha_bbdd` se convierte de formato `AAAAMMDD` a formato fecha real `AAAA-MM-DD`.
7. No se aplica escalamiento ni one-hot encoding en F2: esas transformaciones dependen del algoritmo que se seleccione en F3. Mantener las variables interpretables evita introducir una decisión de modelamiento antes de tiempo.

In [5]:
COLUMNAS_NUMERICAS = [
    "rbd", "dvrbd",
    "cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd", "cod_deprov_rbd",
    "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_mate4b_rbd", "prom_mate4b_rbd",
    "marca_mate4b_rbd", "noaplica", "agno",
]

COLUMNAS_TEXTO = [
    "nom_rbd", "nom_deprov_rbd", "codigo_bbdd", "grado",
]


def convertir_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte numéricos y fecha; falla ante valores no convertibles no nulos."""
    out = df.copy()

    for col in COLUMNAS_NUMERICAS:
        convertida = pd.to_numeric(out[col], errors="coerce")
        invalidos = out[col].notna() & convertida.isna()
        if invalidos.any():
            ejemplos = out.loc[invalidos, col].astype(str).head(5).tolist()
            raise ValueError(
                f"Valores no numéricos en {col}: {int(invalidos.sum())}. "
                f"Ejemplos: {ejemplos}"
            )
        out[col] = convertida

    fecha_num = pd.to_numeric(out["fecha_bbdd"], errors="coerce").astype("Int64")
    fecha_texto = fecha_num.astype("string")
    out["fecha_bbdd"] = pd.to_datetime(
        fecha_texto,
        format="%Y%m%d",
        errors="coerce",
    )

    if out["fecha_bbdd"].isna().any():
        raise ValueError(
            f"fecha_bbdd contiene {int(out['fecha_bbdd'].isna().sum())} "
            "valores inválidos o vacíos."
        )

    return out


def limpiar_textos(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza campos de texto sin modificar semánticamente su contenido."""
    out = df.copy()
    for col in COLUMNAS_TEXTO:
        out[col] = (
            out[col]
            .astype("string")
            .str.replace(r"[\x00-\x1F\x7F]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace("", pd.NA)
        )
    return out


df_limpio = limpiar_textos(convertir_tipos(df_raw))

print("Tipos y textos normalizados.")
print("Fecha(s) de base:", df_limpio["fecha_bbdd"].dt.strftime("%Y-%m-%d").unique().tolist())

Tipos y textos normalizados.
Fecha(s) de base: ['2026-06-22']


## 6. Transformación: categorías, geografía y efectividad

Esta etapa:

- traduce dependencia, GSE y ruralidad;
- cruza las claves geográficas con la `DimGeografia` embebida;
- determina efectividad;
- conserva la causa de exclusión **antes** de filtrar;
- evita cambios silenciosos en el número de filas durante el `merge`.

Los nombres de región, provincia y comuna finales provienen de `DimGeografia`, no de nombres libres del archivo SIMCE.

In [6]:
CLAVES_GEO = ["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]


def validar_codigos_catalogo(df: pd.DataFrame) -> None:
    """Detiene el proceso si aparecen códigos categóricos desconocidos."""
    controles = [
        ("cod_depe1", DEPENDENCIA_1),
        ("cod_depe2", DEPENDENCIA_2),
        ("cod_grupo", GSE),
        ("cod_rural_rbd", RURALIDAD),
    ]
    for columna, catalogo in controles:
        desconocidos = sorted(
            set(df[columna].dropna().astype(int).tolist()) - set(catalogo.keys())
        )
        if desconocidos:
            raise ValueError(
                f"Códigos sin catálogo en {columna}: {desconocidos}"
            )

    marcas_desconocidas = sorted(
        set(df["marca_mate4b_rbd"].dropna().astype(int).tolist())
        - set(OBS_PUNTAJE.keys())
    )
    if marcas_desconocidas:
        raise ValueError(
            "Aparecieron marcas de puntaje no documentadas: "
            f"{marcas_desconocidas}"
        )


def agregar_categorias(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega etiquetas categóricas conservando los códigos originales."""
    validar_codigos_catalogo(df)
    out = df.copy()
    out["dependencia_6_cat"] = out["cod_depe1"].map(DEPENDENCIA_1)
    out["dependencia_4_cat"] = out["cod_depe2"].map(DEPENDENCIA_2)
    out["grupo_socioeconomico"] = out["cod_grupo"].map(GSE)
    out["ruralidad"] = out["cod_rural_rbd"].map(RURALIDAD)
    return out


def agregar_geografia(
    df: pd.DataFrame,
    dim_geografia: pd.DataFrame,
) -> pd.DataFrame:
    """Enriquece el dataset mediante un JOIN many-to-one validado."""
    out = df.copy()

    for col in CLAVES_GEO:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

    filas_antes = len(out)
    out = out.merge(
        dim_geografia,
        how="left",
        on=CLAVES_GEO,
        validate="many_to_one",
    )

    if len(out) != filas_antes:
        raise AssertionError(
            "El JOIN geográfico modificó la cantidad de registros."
        )

    sin_geo = out["comuna"].isna()
    if sin_geo.any():
        claves = (
            out.loc[sin_geo, CLAVES_GEO]
            .drop_duplicates()
            .sort_values(CLAVES_GEO)
        )
        display(claves)
        raise ValueError(
            f"Hay {int(sin_geo.sum())} registros sin coincidencia en DimGeografia."
        )

    return out


def clasificar_efectividad(df: pd.DataFrame) -> pd.DataFrame:
    """Clasifica Efectiva/No Efectiva y explicita el motivo de exclusión."""
    out = df.copy()

    out["obs_puntaje"] = out["marca_mate4b_rbd"].map(OBS_PUNTAJE)

    tiene_alumnos = out["nalu_mate4b_rbd"].fillna(0).gt(0)
    sin_observacion = out["obs_puntaje"].isna()

    out["efectividad"] = "No Efectiva"
    out.loc[tiene_alumnos & sin_observacion, "efectividad"] = "Efectiva"

    out["motivo_exclusion"] = "Registro efectivo"

    sin_alumnos = ~tiene_alumnos
    con_observacion = ~sin_observacion

    out.loc[
        sin_alumnos & ~con_observacion,
        "motivo_exclusion",
    ] = "Sin alumnos evaluados"

    out.loc[
        ~sin_alumnos & con_observacion,
        "motivo_exclusion",
    ] = "Observación de puntaje"

    out.loc[
        sin_alumnos & con_observacion,
        "motivo_exclusion",
    ] = "Sin alumnos y con observación de puntaje"

    return out


df_transformado = agregar_categorias(df_limpio)
df_transformado = agregar_geografia(df_transformado, dim_geo)
df_transformado = clasificar_efectividad(df_transformado)

print(df_transformado["efectividad"].value_counts(dropna=False))

efectividad
Efectiva       6524
No Efectiva     619
Name: count, dtype: int64


## 7. Auditoría del filtro y análisis de materialidad

Antes de aplicar el filtro definitivo sobre los registros SIMCE, se realiza una auditoría del conjunto de datos con el propósito de **cuantificar el efecto de las exclusiones, justificar las decisiones adoptadas y mantener trazabilidad sobre los registros que no forman parte de la base analítica final**.

Los registros clasificados como **No Efectivos no son eliminados de manera silenciosa**. Previamente se identifican y caracterizan para evaluar las razones de su exclusión y su posible incidencia sobre el conjunto de datos.

La auditoría considera, entre otros, los siguientes elementos:

* cantidad de registros originales;
* cantidad de registros Efectivos y No Efectivos;
* registros que poseen puntaje disponible;
* registros No Efectivos que conservan puntaje;
* cantidad de estudiantes asociados al conjunto original y a los registros excluidos;
* promedio del puntaje de los registros Efectivos;
* promedio considerando todos los registros que poseen puntaje;
* diferencia entre ambos promedios;
* distribución de los registros según marca y motivo de exclusión;
* distribución territorial de los registros excluidos.

### 7.1 Criterio de conservación y exclusión

La disponibilidad de un dato no se considera, por sí sola, suficiente para determinar su utilidad analítica.

Para formar parte del conjunto procesado, un registro debe cumplir los criterios definidos de **efectividad y representatividad**. En consecuencia, un establecimiento puede disponer de un puntaje registrado y, aun así, ser excluido cuando la marca asociada indica que dicho resultado no representa adecuadamente el desempeño que se busca estudiar.

Por esta razón, el criterio de exclusión no se basa únicamente en la existencia o ausencia de `prom_mate4b_rbd`, sino también en la información contenida en `marca_mate4b_rbd` y su correspondiente descripción normalizada.

Los registros clasificados como No Efectivos se conservan dentro de la auditoría para mantener su trazabilidad, pero **no se incorporan al dataset analítico final**, debido a que sus condiciones de aplicación o representatividad reducen su utilidad para responder la problemática definida en el proyecto.

### 7.2 Análisis de materialidad

El análisis de materialidad busca determinar la relevancia que tienen los registros excluidos respecto del conjunto original, sin utilizar un porcentaje arbitrario como umbral para decidir si una exclusión es importante o no.

La evaluación se realiza considerando de manera conjunta tres dimensiones:

1. **Cantidad absoluta de registros y estudiantes excluidos**, para dimensionar el volumen de información afectada.
2. **Diferencia entre los promedios de puntaje**, comparando los registros Efectivos con el conjunto total de registros que poseen puntaje.
3. **Concentración territorial de las exclusiones**, para identificar si los registros descartados se distribuyen de manera generalizada o se concentran en determinadas regiones o territorios.

La diferencia entre el promedio de los registros Efectivos y el promedio de todos los registros con puntaje se utiliza como una **medida de sensibilidad del filtro**. Su objetivo es evaluar cuánto cambia el comportamiento general del puntaje al aplicar los criterios de representatividad.

Esta comparación no se utiliza como criterio para reincorporar registros excluidos. Un impacto reducido sobre el promedio general no convierte un resultado No Efectivo en representativo, del mismo modo que una diferencia mayor no implica automáticamente que el registro deba conservarse.

### 7.3 Tratamiento de registros con marca 2

Se presta especial atención a los registros con `marca_mate4b_rbd = 2`.

La normalización de las marcas permite interpretar estos casos como resultados que, por causas ajenas a la Agencia de Calidad de la Educación, **no son considerados representativos del desempeño de los estudiantes evaluados**.

En algunos de estos registros puede existir un valor numérico en `prom_mate4b_rbd`. Sin embargo, la existencia del puntaje no elimina la restricción asociada a su representatividad.

Para los objetivos del presente proyecto, cuyo análisis posterior requiere comparar resultados académicos entre establecimientos y características territoriales o socioeconómicas, incorporar observaciones identificadas explícitamente como no representativas podría introducir información cuya interpretación no es equivalente a la de los registros Efectivos.

Por esta razón, estos casos:

* permanecen identificados dentro de la auditoría;
* mantienen disponible su información original para fines de trazabilidad;
* no son imputados ni modificados;
* y son excluidos del conjunto analítico final.

De esta forma se distingue entre **disponibilidad del dato** y **validez del dato para el propósito específico del análisis**.

### 7.4 Decisión sobre valores faltantes e imputación

No se realiza imputación del puntaje promedio para los registros que no cuentan con un resultado válido.

Esta decisión responde a que la ausencia de información en estos casos no corresponde necesariamente a un problema técnico de pérdida de datos que pueda solucionarse mediante media, mediana u otro mecanismo de imputación.

En parte de los registros, la ausencia o invalidación del resultado está relacionada con condiciones de aplicación, número de estudiantes evaluados o restricciones de representatividad establecidas en la propia fuente.

Asignar artificialmente un puntaje a estos casos podría:

* introducir valores que no fueron observados;
* reducir artificialmente la variabilidad del conjunto;
* alterar comparaciones entre establecimientos;
* y otorgar validez analítica a registros que originalmente no cumplen los criterios de representatividad.

Por lo anterior, se opta por **no imputar puntajes** y mantener separados los registros que no cumplen los requisitos establecidos.

### 7.5 Componente territorial de la auditoría

Debido a que la problemática del proyecto considera diferencias territoriales en los resultados SIMCE, la auditoría también analiza la distribución regional de los registros excluidos.

Este control permite detectar si las exclusiones presentan una concentración territorial relevante. Esta revisión es importante porque una eliminación desproporcionada de establecimientos pertenecientes a una determinada región podría modificar la cobertura territorial del conjunto procesado y afectar posteriormente las comparaciones geográficas.

La dimensión territorial se utiliza, por tanto, como un **control de cobertura y representatividad del dataset procesado**, y no como un criterio independiente para reincorporar observaciones que no cumplen las condiciones de efectividad.

### 7.6 Decisión metodológica final

El filtrado aplicado busca priorizar la **calidad y representatividad de las observaciones por sobre la conservación indiscriminada de registros**.


In [13]:
def construir_auditoria(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Construye auditoría detallada y resúmenes antes de filtrar."""

    total = len(df)

    efectiva = df["efectividad"].eq("Efectiva")
    no_efectiva = ~efectiva
    con_puntaje = df["prom_mate4b_rbd"].notna()
    no_efectiva_con_puntaje = no_efectiva & con_puntaje

    # --------------------------------------------------------
    # Promedios para análisis de materialidad
    # --------------------------------------------------------

    promedio_efectivos = df.loc[
        efectiva,
        "prom_mate4b_rbd",
    ].mean()

    promedio_todos_con_puntaje = df.loc[
        con_puntaje,
        "prom_mate4b_rbd",
    ].mean()

    diferencia_promedio = (
        promedio_todos_con_puntaje - promedio_efectivos
        if pd.notna(promedio_efectivos)
        and pd.notna(promedio_todos_con_puntaje)
        else pd.NA
    )

    # --------------------------------------------------------
    # Resumen general
    # --------------------------------------------------------

    resumen = pd.DataFrame({
        "indicador": [
            "Registros originales",
            "Registros efectivos",
            "Registros no efectivos",
            "Registros con puntaje",
            "No efectivos con puntaje",
            "Alumnos asociados al bruto",
            "Alumnos asociados a no efectivos",
            "Promedio puntaje - efectivos",
            "Promedio puntaje - todos con puntaje",
            "Diferencia de promedio (puntos)",
        ],
        "valor": [
            total,
            int(efectiva.sum()),
            int(no_efectiva.sum()),
            int(con_puntaje.sum()),
            int(no_efectiva_con_puntaje.sum()),

            float(
                df["nalu_mate4b_rbd"]
                .fillna(0)
                .sum()
            ),

            float(
                df.loc[
                    no_efectiva,
                    "nalu_mate4b_rbd",
                ]
                .fillna(0)
                .sum()
            ),

            (
                round(
                    float(promedio_efectivos),
                    4,
                )
                if pd.notna(promedio_efectivos)
                else pd.NA
            ),

            (
                round(
                    float(promedio_todos_con_puntaje),
                    4,
                )
                if pd.notna(promedio_todos_con_puntaje)
                else pd.NA
            ),

            (
                round(
                    float(diferencia_promedio),
                    4,
                )
                if pd.notna(diferencia_promedio)
                else pd.NA
            ),
        ],
    })

    # --------------------------------------------------------
    # Auditoría de registros excluidos
    # --------------------------------------------------------

    columnas_auditoria = [
        "rbd",
        "dvrbd",
        "nom_rbd",

        "cod_reg_rbd",
        "region",

        "cod_pro_rbd",
        "provincia",

        "cod_com_rbd",
        "comuna",

        "nalu_mate4b_rbd",
        "prom_mate4b_rbd",

        "marca_mate4b_rbd",
        "obs_puntaje",
        "motivo_exclusion",

        "noaplica",
        "codigo_bbdd",
        "fecha_bbdd",
        "grado",
        "agno",
    ]

    auditoria = df.loc[
        no_efectiva,
        columnas_auditoria,
    ].copy()

    # --------------------------------------------------------
    # Identificación de marca 2 con puntaje
    # --------------------------------------------------------
    # Se conserva solo como información de auditoría.
    # No genera warning.
    # --------------------------------------------------------

    auditoria["marca_2_con_puntaje"] = (
        auditoria["marca_mate4b_rbd"].eq(2)
        & auditoria["prom_mate4b_rbd"].notna()
    )

    # --------------------------------------------------------
    # Resumen por marca
    # --------------------------------------------------------

    resumen_marcas = (
        df.groupby(
            [
                "marca_mate4b_rbd",
                "obs_puntaje",
            ],
            dropna=False,
        )
        .agg(
            registros=(
                "rbd",
                "size",
            ),
            alumnos=(
                "nalu_mate4b_rbd",
                "sum",
            ),
            con_puntaje=(
                "prom_mate4b_rbd",
                "count",
            ),
        )
        .reset_index()
        .sort_values(
            "registros",
            ascending=False,
        )
    )

    return (
        auditoria,
        resumen,
        resumen_marcas,
    )


# ============================================================
# EJECUCIÓN
# ============================================================

auditoria_filtro, resumen_filtro, resumen_marcas = (
    construir_auditoria(
        df_transformado
    )
)


display(resumen_filtro)

display(resumen_marcas)


print(
    "Casos marca 2 con puntaje:",
    int(
        auditoria_filtro[
            "marca_2_con_puntaje"
        ].sum()
    )
)

,indicador,valor
0,Registros originales,7143.0000
1,Registros efectivos,6524.0000
2,Registros no efectivos,619.0000
3,Registros con puntaje,6579.0000
4,No efectivos con puntaje,55.0000
5,Alumnos asociados al bruto,213967.0000
6,Alumnos asociados a no efectivos,1299.0000
7,Promedio puntaje - efectivos,254.6971
8,Promedio puntaje - todos con puntaje,254.7348
9,Diferencia de promedio (puntos),0.0376


,marca_mate4b_rbd,obs_puntaje,registros,alumnos,con_puntaje
2,NaN,NaN,6638,212668,6524
0,1.0,"No es posible reportar resultados, porque la c...",450,439,0
1,2.0,"Por causas ajenas a la Agencia, los resultados...",55,860,55


Casos marca 2 con puntaje: 55


## 8. Pruebas técnicas — casos normal, límite y excepción

La validación no se limita al dataset real. Se prueban reglas con datos sintéticos controlados:

- **Caso normal:** alumnos > 0, sin marca → Efectiva.
- **Caso límite:** 0 alumnos → No Efectiva.
- **Caso con observación:** alumnos > 0 y marca documentada → No Efectiva.
- **Caso marca 2 con puntaje:** debe quedar No Efectiva y ser auditable.
- **Excepción:** código de marca desconocido → el proceso debe detenerse.

Esto cubre explícitamente la exigencia de la rúbrica sobre casos normales, límites y excepciones.

In [8]:
def ejecutar_pruebas_regla_efectividad() -> None:
    """Pruebas pequeñas y reproducibles de las reglas críticas de F2."""
    prueba = pd.DataFrame({
        "rbd": [1, 2, 3, 4],
        "nalu_mate4b_rbd": [20, 0, 15, 12],
        "prom_mate4b_rbd": [250, pd.NA, pd.NA, 260],
        "marca_mate4b_rbd": [pd.NA, pd.NA, 1, 2],
    })

    resultado = clasificar_efectividad(prueba)

    assert resultado.loc[0, "efectividad"] == "Efectiva"
    assert resultado.loc[1, "efectividad"] == "No Efectiva"
    assert resultado.loc[2, "efectividad"] == "No Efectiva"
    assert resultado.loc[3, "efectividad"] == "No Efectiva"

    # Excepción: una marca desconocida debe ser detectada por la validación de catálogo.
    prueba_error = pd.DataFrame({
        "cod_depe1": [1],
        "cod_depe2": [1],
        "cod_grupo": [1],
        "cod_rural_rbd": [1],
        "marca_mate4b_rbd": [99],
    })

    try:
        validar_codigos_catalogo(prueba_error)
    except ValueError:
        paso_excepcion = True
    else:
        paso_excepcion = False

    assert paso_excepcion, "La prueba de marca desconocida no generó la excepción esperada."

    print("✓ Caso normal validado")
    print("✓ Caso límite validado")
    print("✓ Caso con observación validado")
    print("✓ Caso marca 2 con puntaje validado")
    print("✓ Excepción por código desconocido validada")


ejecutar_pruebas_regla_efectividad()

✓ Caso normal validado
✓ Caso límite validado
✓ Caso con observación validado
✓ Caso marca 2 con puntaje validado
✓ Excepción por código desconocido validada


## 9. Filtrado y estandarización del producto final

Solo después del diagnóstico y la auditoría se conservan registros **Efectivos**.

Las columnas auxiliares utilizadas para decidir el filtro no se necesitan en el dataset final porque quedan preservadas en `auditoria_filtro.csv`.

Las variables de estándares de aprendizaje porcentuales fueron eliminadas del F2 por decisión de alcance y **no se cargan ni transforman**.

In [9]:
RENOMBRE_FINAL = {
    "dvrbd": "dv_rbd",
    "nom_rbd": "nombre_establecimiento",
    "nom_deprov_rbd": "deprov",
    "nalu_mate4b_rbd": "n_alumnos",
    "prom_mate4b_rbd": "puntaje_promedio",
    "noaplica": "no_aplica",
    "agno": "anio",
}

COLUMNAS_FINALES = [
    "rbd", "dv_rbd", "nombre_establecimiento", "asignatura",

    "cod_reg_rbd", "region",
    "cod_pro_rbd", "provincia",
    "cod_com_rbd", "comuna",
    "deprov",

    "pais",
    "ubicacion_region",
    "ubicacion_provincia",
    "ubicacion_comuna",
    "Zona", "Macrozona", "Orden",

    "dependencia_6_cat",
    "dependencia_4_cat",
    "grupo_socioeconomico",
    "ruralidad",

    "n_alumnos",
    "puntaje_promedio",

    "efectividad",
    "no_aplica",
    "codigo_bbdd",
    "fecha_bbdd",
    "grado",
    "anio",
]


def construir_dataset_final(df: pd.DataFrame) -> pd.DataFrame:
    """Filtra efectivos, renombra y selecciona exclusivamente el esquema final."""
    out = df.loc[df["efectividad"].eq("Efectiva")].copy()
    out = out.rename(columns=RENOMBRE_FINAL)

    out["asignatura"] = "Matematica"

    # Tipos finales
    for col in [
        "rbd", "dv_rbd",
        "cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd",
        "n_alumnos", "anio", "Orden",
    ]:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

    out["puntaje_promedio"] = pd.to_numeric(
        out["puntaje_promedio"],
        errors="coerce",
    )

    out["no_aplica"] = (
        pd.to_numeric(out["no_aplica"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )

    return out[COLUMNAS_FINALES].copy()


df_final = construir_dataset_final(df_transformado)

print(
    f"Dataset final construido: "
    f"{df_final.shape[0]:,} filas x {df_final.shape[1]} columnas"
)

Dataset final construido: 6,524 filas x 30 columnas


## 10. Cobertura regional

`cobertura_regional.csv` permite comprobar que el filtro no afecta de forma invisible a determinadas regiones.

Se informan **conteos absolutos** por región:

- registros originales;
- registros efectivos;
- registros no efectivos;
- excluidos que conservaban puntaje;
- alumnos del bruto;
- alumnos efectivos;
- alumnos asociados a no efectivos.

Este archivo es evidencia de F2; no constituye todavía una priorización territorial.

In [10]:
def construir_cobertura_regional(df: pd.DataFrame) -> pd.DataFrame:
    """Resume la cobertura antes y después del filtro por región."""
    trabajo = df.copy()
    trabajo["es_efectiva"] = trabajo["efectividad"].eq("Efectiva")
    trabajo["es_no_efectiva"] = ~trabajo["es_efectiva"]
    trabajo["no_efectiva_con_puntaje"] = (
        trabajo["es_no_efectiva"]
        & trabajo["prom_mate4b_rbd"].notna()
    )
    trabajo["alumnos_efectivos"] = (
        trabajo["nalu_mate4b_rbd"].fillna(0)
        * trabajo["es_efectiva"].astype(int)
    )
    trabajo["alumnos_no_efectivos"] = (
        trabajo["nalu_mate4b_rbd"].fillna(0)
        * trabajo["es_no_efectiva"].astype(int)
    )

    cobertura = (
        trabajo.groupby(
            ["cod_reg_rbd", "region", "Zona", "Macrozona", "Orden"],
            dropna=False,
        )
        .agg(
            registros_originales=("rbd", "size"),
            registros_efectivos=("es_efectiva", "sum"),
            registros_no_efectivos=("es_no_efectiva", "sum"),
            no_efectivos_con_puntaje=("no_efectiva_con_puntaje", "sum"),
            alumnos_bruto=("nalu_mate4b_rbd", "sum"),
            alumnos_efectivos=("alumnos_efectivos", "sum"),
            alumnos_no_efectivos=("alumnos_no_efectivos", "sum"),
        )
        .reset_index()
        .sort_values("Orden")
    )

    return cobertura


cobertura_regional = construir_cobertura_regional(df_transformado)
display(cobertura_regional)

,cod_reg_rbd,region,Zona,Macrozona,Orden,registros_originales,registros_efectivos,registros_no_efectivos,no_efectivos_con_puntaje,alumnos_bruto,alumnos_efectivos,alumnos_no_efectivos
14,15,Arica y Parinacota,Norte,Macrozona Norte,1,81,69,12,1,3023,2946,77
0,1,Tarapacá,Norte,Macrozona Norte,2,109,102,7,0,5295,5290,5
1,2,Antofagasta,Norte,Macrozona Norte,3,136,134,2,0,8307,8305,2
2,3,Atacama,Norte,Macrozona Norte,4,110,102,8,0,4082,4076,6
3,4,Coquimbo,Norte,Macrozona Centro Norte,5,447,372,75,2,10187,10106,81
4,5,Valparaíso,Centro,Macrozona Centro Norte,6,759,740,19,3,21403,21361,42
12,13,Metropolitana de Santiago,Centro,Macrozona Metropolitana,7,1754,1723,31,16,82485,82209,276
5,6,Libertador General Bernardo O'Higgins,Centro,Macrozona Centro Norte,8,445,421,24,5,11885,11798,87
6,7,Maule,Centro,Macrozona Centro Sur,9,558,505,53,6,13615,13507,108
15,16,Ñuble,Centro,Macrozona Centro Sur,10,290,251,39,2,5495,5436,59


## 11. Validaciones finales de integridad y coherencia

Las validaciones comprueban:

- dataset no vacío;
- solo Matemática;
- solo registros Efectivos;
- alumnos evaluados mayores que cero;
- RBD no nulo y único;
- puntaje y geografía completos;
- códigos territoriales completos;
- fecha válida;
- ausencia de columnas duplicadas;
- reconciliación entre bruto, efectivos y auditoría;
- reconciliación de cobertura regional;
- ausencia de las variables porcentuales eliminadas del alcance.

In [11]:
def validar_dataset_final(
    df_final: pd.DataFrame,
    df_transformado: pd.DataFrame,
    auditoria: pd.DataFrame,
    cobertura: pd.DataFrame,
) -> dict[str, bool]:
    """Valida integridad, consistencia y reconciliación del resultado de F2."""
    validaciones = {
        "Dataset final no vacío":
            not df_final.empty,

        "Asignatura = Matematica":
            bool(df_final["asignatura"].eq("Matematica").all()),

        "Efectividad = Efectiva":
            bool(df_final["efectividad"].eq("Efectiva").all()),

        "n_alumnos > 0":
            bool(df_final["n_alumnos"].gt(0).all()),

        "RBD no nulo":
            bool(df_final["rbd"].notna().all()),

        "RBD único":
            bool(df_final["rbd"].is_unique),

        "Puntaje no nulo":
            bool(df_final["puntaje_promedio"].notna().all()),

        "Geografía completa":
            bool(
                df_final[
                    [
                        "cod_reg_rbd", "region",
                        "cod_pro_rbd", "provincia",
                        "cod_com_rbd", "comuna",
                        "Zona", "Macrozona", "Orden",
                    ]
                ].notna().all().all()
            ),

        "fecha_bbdd válida":
            bool(df_final["fecha_bbdd"].notna().all()),

        "Sin columnas duplicadas":
            bool(not df_final.columns.duplicated().any()),

        "Reconciliación bruto = final + auditoría":
            len(df_transformado) == len(df_final) + len(auditoria),

        "Cobertura regional reconcilia con bruto":
            int(cobertura["registros_originales"].sum())
            == len(df_transformado),

        "Cobertura regional reconcilia con final":
            int(cobertura["registros_efectivos"].sum())
            == len(df_final),

        "Sin columnas pct eliminadas":
            not any(
                col in df_final.columns
                for col in [
                    "pct_insuficiente",
                    "pct_elemental",
                    "pct_adecuado",
                ]
            ),
    }

    fallidas = [
        nombre
        for nombre, cumple in validaciones.items()
        if not cumple
    ]

    if fallidas:
        raise AssertionError(
            "Fallaron las validaciones: " + "; ".join(fallidas)
        )

    return validaciones


CRITERIOS_CUMPLIDOS = validar_dataset_final(
    df_final,
    df_transformado,
    auditoria_filtro,
    cobertura_regional,
)

print("=" * 72)
print("VALIDACIONES FINALES")
print("=" * 72)
for criterio, cumple in CRITERIOS_CUMPLIDOS.items():
    print(f"✓ {criterio}" if cumple else f"✗ {criterio}")

VALIDACIONES FINALES
✓ Dataset final no vacío
✓ Asignatura = Matematica
✓ Efectividad = Efectiva
✓ n_alumnos > 0
✓ RBD no nulo
✓ RBD único
✓ Puntaje no nulo
✓ Geografía completa
✓ fecha_bbdd válida
✓ Sin columnas duplicadas
✓ Reconciliación bruto = final + auditoría
✓ Cobertura regional reconcilia con bruto
✓ Cobertura regional reconcilia con final
✓ Sin columnas pct eliminadas


## 12. Exportación y verificación de ida y vuelta

Se generan tres productos reales:

1. `simce4b2025_matematica_efectiva_AAAAMMDDHHMM.csv`  
   Dataset limpio y validado para F3.
2. `auditoria_filtro.csv`  
   Registros No Efectivos y causa de exclusión.
3. `cobertura_regional.csv`  
   Reconciliación territorial antes/después del filtro.

Después de exportar el dataset principal, se vuelve a leer y se compara con el DataFrame original para detectar alteraciones de escritura.

In [12]:
def exportar_resultados(
    df_final: pd.DataFrame,
    auditoria: pd.DataFrame,
    cobertura: pd.DataFrame,
) -> None:
    """Exporta los tres productos de F2 y verifica el archivo principal."""
    df_final.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
    )

    auditoria.to_csv(
        AUDITORIA_FILE,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
    )

    cobertura.to_csv(
        COBERTURA_FILE,
        index=False,
        encoding="utf-8-sig",
    )

    # Verificación de ida y vuelta del dataset principal.
    releido = pd.read_csv(
        OUTPUT_FILE,
        encoding="utf-8-sig",
    )
    releido["fecha_bbdd"] = pd.to_datetime(
        releido["fecha_bbdd"],
        format="%Y-%m-%d",
        errors="raise",
    )
    releido["no_aplica"] = (
        pd.to_numeric(releido["no_aplica"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )

    pd.testing.assert_frame_equal(
        releido,
        df_final.reset_index(drop=True),
        check_dtype=False,
    )


exportar_resultados(
    df_final,
    auditoria_filtro,
    cobertura_regional,
)

print("=" * 72)
print("RESULTADO F2")
print("=" * 72)
print(f"Filas originales              : {len(df_transformado):,}")
print(f"Filas válidas / efectivas     : {len(df_final):,}")
print(f"Filas no efectivas auditadas  : {len(auditoria_filtro):,}")
print(
    "No efectivas con puntaje      : "
    f"{int(auditoria_filtro['prom_mate4b_rbd'].notna().sum()):,}"
)
print(
    "Marca 2 con puntaje auditada  : "
    f"{int(auditoria_filtro['marca_2_con_puntaje'].sum()):,}"
)
print(f"Archivo principal             : {OUTPUT_FILE.name}")
print(f"Auditoría                     : {AUDITORIA_FILE.name}")
print(f"Cobertura regional            : {COBERTURA_FILE.name}")

RESULTADO F2
Filas originales              : 7,143
Filas válidas / efectivas     : 6,524
Filas no efectivas auditadas  : 619
No efectivas con puntaje      : 55
Marca 2 con puntaje auditada  : 55
Archivo principal             : simce4b2025_matematica_efectiva_202609142338.csv
Auditoría                     : auditoria_filtro.csv
Cobertura regional            : cobertura_regional.csv


## 13. Conclusión técnica para F2

La fase deja un conjunto de datos procesado **sin imputar puntajes, identidades ni territorio**. Las exclusiones se realizan únicamente después de medir y documentar sus causas.

La auditoría permite distinguir dos preguntas diferentes:

- **¿Cuántos establecimientos salen de la base?**
- **¿Cuántos de los excluidos todavía tenían puntaje?**

Esa distinción es importante para discutir materialidad sin asumir que todo registro excluido afecta de la misma forma el análisis posterior. La diferencia entre el promedio de efectivos y el promedio de todos los registros con puntaje se muestra como prueba de sensibilidad, pero **no se utiliza para reincorporar registros cuya representatividad está observada**.

Los casos `marca_mate4b_rbd = 2` con puntaje se preservan en la auditoría para que la decisión pueda verificarse contra el diccionario oficial 2025 antes de la entrega final.

### Tratamientos deliberadamente NO aplicados
- No se eliminan atípicos solo por IQR.
- No se imputa puntaje promedio.
- No se imputa geografía.
- No se escalan variables todavía.
- No se codifican categorías mediante one-hot todavía.
- No se realizan gráficos ni inferencia estadística en F2.

## 14. Vinculación con el mapa conceptual y fases del proyecto

| Elemento | Estado | Evidencia |
|---|---|---|
| Fuente oficial SIMCE 2025 | Implementado | `data/raw/` + carga selectiva |
| Exploración y calidad | Implementado en F2 | diagnóstico de tipos, nulos, duplicados y atípicos |
| Limpieza | Implementado en F2 | casting, fecha, limpieza de textos |
| Transformación | Implementado en F2 | categorías + `DimGeografia` embebida |
| Control de representatividad | Implementado en F2 | efectividad + `auditoria_filtro.csv` |
| Validación técnica | Implementado en F2 | asserts + pruebas normal/límite/excepción |
| Cobertura territorial | Implementado como control F2 | `cobertura_regional.csv` |
| Análisis estadístico / significancia | Proyectado | Fase 3 |
| Escalamiento/encoding para algoritmo | Proyectado | Fase 3, según modelo elegido |
| Visualización y comunicación de resultados | Proyectado | fase posterior |

Esta tabla evita afirmar que componentes posteriores ya están desarrollados y deja trazabilidad directa entre planificación e implementación.

## 15. Checklist previo a entrega

Antes de subir el notebook definitivo:

- ejecutar **Restart Kernel → Run All Cells**;
- comprobar que la numeración de ejecución sea continua;
- **no borrar los outputs**;
- verificar que `data/processed/` contenga los tres productos de F2;
- adjuntar/conservar respaldo del diccionario oficial SIMCE 2025 utilizado para `marca_mate4b_rbd`, especialmente el código 2;
- comprobar que README explique cómo ejecutar F1 y F2;
- comprobar que `requirements.txt` tenga versiones fijadas;
- citar en el informe las cifras producidas por `resumen_filtro`, no cifras escritas manualmente.